# LG Aimers 7기 · 식음업장 메뉴 수요 예측

리조트 내 다양한 식음업장의 **일별 메뉴 판매량(28일 입력 → 7일 예측, Direct 멀티-호라이즌)** 을 LightGBM Tweedie 회귀로 모델링한다.

## 1. 문제 정의
- **입력 (X)** : 메뉴별 최근 28일 판매량 시퀀스
- **출력 (y)** : 동일 메뉴의 향후 7일(`T+1` ~ `T+7`) 판매량
- **평가지표** : 업장 동일가중 SMAPE

데이터는 *공휴일·이벤트일 폭증* 과 *희소(거의 안 팔리는) 메뉴* 가 공존해 일반 시계열 모델로는 정확도 확보가 어렵다.

## 2. 아키텍처 개요

```
[28일 메뉴별 시퀀스]
      │
      ├──▶ window_features      (lag1~7, 평균/표준편차/추세 기울기, 추세비)
      ├──▶ horizon_context(h)   (예측 시점 요일·주말 플래그, 동요일 평균, lag7)
      │
      ▼
[Tweedie LGBMRegressor × 7]   ← Horizon별 Direct 모델 (비재귀)
      │
      ▼
[모델/단순 lag7/동요일 평균 가중 블렌딩]
      W_MODEL=0.6 · W_LAG7=0.3 · W_MEAN=0.1
      │
      ▼
[zero floor (1e-6)] → submission.csv
```

## 3. 핵심 설계 포인트
| 포인트 | 설명 |
| --- | --- |
| **Tweedie 손실** | `tweedie_variance_power=1.5` → 0이 많고 양수 꼬리가 긴 판매량 분포에 적합 |
| **Direct 7-스텝** | 호라이즌(h)별로 별도 모델 학습 → 재귀예측 오차 누적 방지 |
| **하이브리드 블렌딩** | 모델 예측 + 단순 lag7 + 동요일 평균을 6:3:1로 가중 평균 → 과적합·노이즈 동시 완화 |
| **샘플 가중** | `y>0`에 1.0, `y=0`에 0.2 → SMAPE 산식 정렬(영구간 과도 학습 억제) |
| **Zero Floor** | 0 예측을 `1e-6`로 치환 → SMAPE 분모 0 회피 |


## 4. 라이브러리 임포트

In [ ]:
import os, gc
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor

## 5. 경로 / 상수 설정

- `LOOKBACK = 28` : 입력 윈도우 길이 (4주)
- `HORIZON  = 7`  : 예측 길이 (1주)
- `EPS = 1e-6`    : 0 예측 방지용 floor

데이터(`train/`, `test/`, `sample_submission.csv`)는 노트북의 **상위 폴더** 인 `/Users/jaemin/Desktop/open` 에 위치한다.

In [ ]:
BASE_DIR = '/Users/jaemin/Desktop/open'
TRAIN_PATH = os.path.join(BASE_DIR, 'train', 'train.csv')
TEST_DIR   = os.path.join(BASE_DIR, 'test')
SAMPLE_SUB = os.path.join(BASE_DIR, 'sample_submission.csv')

OUTPUT_DIR = os.path.join(BASE_DIR, 'lg_aimers_demand_prediction')
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOOKBACK = 28
HORIZON  = 7
EPS = 1e-6

## 6. 피처 엔지니어링

각 28일 윈도우(`last28_qty`)에서 두 종류의 피처를 추출한다.

### 6-1. `window_features` — 호라이즌과 무관한 "창 전체" 통계
| 피처 | 의미 |
| --- | --- |
| `lag_1` ~ `lag_7` | 직전 1~7일 판매량 (단기 패턴) |
| `win_mean / std / min / max / sum` | 28일 분포 통계 |
| `last7_mean`, `prev7_mean` | 최근 1주 vs 직전 1주 평균 |
| `win_slope` | 28일 선형회귀 기울기 (추세) |
| `trend_7_7` | `last7_mean / prev7_mean` (추세비) |

### 6-2. `horizon_context(h)` — 예측 시점(`T+h`)별 컨텍스트
| 피처 | 의미 |
| --- | --- |
| `h`, `dow_h`, `is_weekend_h` | 호라이즌·요일·주말 플래그 (휴일/주말 수요 패턴 학습용) |
| `same_dow_mean_h / std_h` | 28일 내 *동일 요일* 의 평균·표준편차 |
| `lag7_for_h` | `T+h-7`의 실측값 (전주 동요일) — 강력한 naive baseline |
| `ratio_lag7_to_same` | `lag7_for_h / same_dow_mean_h` — 동요일 대비 최신 lag7의 강도 |


In [ ]:
def window_features(last28_qty: np.ndarray):
    x = np.asarray(last28_qty, dtype=float)
    feats = {}
    for k in range(1, 8):
        feats[f'lag_{k}'] = float(x[-k]) if len(x) >= k else 0.0
    feats.update({
        'win_mean': float(np.mean(x)),
        'win_std':  float(np.std(x)),
        'win_min':  float(np.min(x)),
        'win_max':  float(np.max(x)),
        'win_sum':  float(np.sum(x)),
        'last7_mean': float(np.mean(x[-7:])),
        'prev7_mean': float(np.mean(x[-14:-7])) if len(x) >= 14 else float(np.mean(x[:-7])) if len(x) > 7 else float(np.mean(x)),
    })
    idx = np.arange(len(x)).reshape(-1, 1)
    lr = LinearRegression().fit(idx, x)
    feats['win_slope'] = float(lr.coef_[0])
    denom = feats['prev7_mean'] if feats['prev7_mean'] != 0 else EPS
    feats['trend_7_7'] = feats['last7_mean'] / denom
    return feats


def horizon_context(last28_dates: np.ndarray, last28_qty: np.ndarray, last_date: pd.Timestamp, h: int):
    d = last_date + pd.Timedelta(days=h)
    dow_h = d.dayofweek
    dows = pd.DatetimeIndex(last28_dates).dayofweek.values
    same_idx = np.where(dows == dow_h)[0]
    same_vals = last28_qty[same_idx] if len(same_idx) else np.array([], dtype=float)
    same_mean = float(np.mean(same_vals)) if len(same_vals) else float(np.mean(last28_qty))
    same_std  = float(np.std(same_vals))  if len(same_vals) else float(np.std(last28_qty))
    naive_lag7 = float(last28_qty[20 + h])
    same_mean_eps = same_mean if same_mean != 0 else EPS
    ratio_lag7_to_same = naive_lag7 / same_mean_eps
    return {
        'h': h,
        'dow_h': dow_h,
        'is_weekend_h': int(dow_h in (5, 6)),
        'same_dow_mean_h': same_mean,
        'same_dow_std_h':  same_std,
        'lag7_for_h': naive_lag7,
        'ratio_lag7_to_same': ratio_lag7_to_same,
    }

## 7. 학습 데이터 로딩

- 음수(환불 등)는 `0`으로 클립 → Tweedie 분포 제약 충족
- 메뉴명을 정수 `menu_id`로 매핑하여 LightGBM이 카테고리 정보를 학습할 수 있도록 함

In [ ]:
train = pd.read_csv(TRAIN_PATH)
train['영업일자'] = pd.to_datetime(train['영업일자'])
train['매출수량'] = train['매출수량'].clip(lower=0)

menus = sorted(train['영업장명_메뉴명'].unique().tolist())
menu2id = {m: i for i, m in enumerate(menus)}
id2menu = {i: m for m, i in menu2id.items()}

print(f'menus = {len(menus)}, rows = {len(train):,}')

## 8. 슬라이딩 윈도우 데이터셋 생성

각 메뉴에 대해 길이 28의 입력 윈도우를 한 칸씩 밀어가며 `(피처, h, y)` 트리플을 생성한다.

- 메뉴 길이가 `LOOKBACK + HORIZON = 35`일 미만이면 학습 대상에서 제외
- 한 윈도우당 `HORIZON=7`개의 학습 샘플 생성 → **"Direct 멀티-호라이즌"** 데이터 구조
- `lastdates_all`을 함께 저장해 **시간 기반 검증 분할**에 사용

In [ ]:
def build_training_windows(df_all: pd.DataFrame):
    X_list, y_list, h_list, lastdate_list, menu_id_list = [], [], [], [], []
    for menu in menus:
        d = df_all[df_all['영업장명_메뉴명'] == menu].sort_values('영업일자')
        if len(d) < LOOKBACK + HORIZON:
            continue
        qty = d['매출수량'].values.astype(float)
        dates = d['영업일자'].values
        for i in range(LOOKBACK - 1, len(dates) - HORIZON):
            last28_qty   = qty[i - LOOKBACK + 1:i + 1]
            last28_dates = dates[i - LOOKBACK + 1:i + 1]
            last_date    = pd.Timestamp(dates[i])
            base = window_features(last28_qty)
            for h in range(1, HORIZON + 1):
                ctx = horizon_context(last28_dates, last28_qty, last_date, h)
                feats = {**base, **ctx, 'menu_id': menu2id[menu]}
                X_list.append(feats)
                y_list.append(float(qty[i + h]))
                h_list.append(h)
                lastdate_list.append(last_date)
                menu_id_list.append(menu2id[menu])
    X = pd.DataFrame(X_list)
    y = np.array(y_list, dtype=float)
    h_arr = np.array(h_list, dtype=int)
    lastdates = pd.to_datetime(pd.Series(lastdate_list))
    mids = np.array(menu_id_list, dtype=int)
    return X, y, h_arr, lastdates, mids


X_all, y_all, h_all, lastdates_all, mids_all = build_training_windows(train)
print(f'X shape = {X_all.shape}, y shape = {y_all.shape}')

## 9. 시간 기반 학습/검증 분할

데이터 누설 방지를 위해 **윈도우의 `last_date` 기준** 으로 분할한다.
- 학습: `last_date ≤ (global_last − 30일)`
- 검증: `last_date >  (global_last − 30일)`  → 최근 1개월

**샘플 가중치(`w`)** :
- `y > 0` → `1.0`
- `y = 0` → `0.2` (수많은 0의 영향을 줄여 SMAPE 산식과 정렬)

In [ ]:
global_last = train['영업일자'].max()
split_date  = global_last - pd.Timedelta(days=30)

tr_idx = (lastdates_all <= split_date).values
va_idx = (lastdates_all >  split_date).values

X_tr, X_va = X_all.iloc[tr_idx].copy(), X_all.iloc[va_idx].copy()
y_tr, y_va = y_all[tr_idx], y_all[va_idx]
h_tr, h_va = h_all[tr_idx], h_all[va_idx]

w_tr = np.where(y_tr > 0, 1.0, 0.2)
w_va = np.where(y_va > 0, 1.0, 0.2)

print(f'train = {tr_idx.sum():,} | valid = {va_idx.sum():,} | split_date = {split_date.date()}')

## 10. Horizon별 Tweedie LGBM 학습

**왜 Direct + Tweedie인가?**
- **Direct** : `h=1..7`을 한 모델에 모두 맡기는 대신 *각 h별로 별도 모델* 을 학습 → 재귀예측 오차가 누적되지 않음
- **Tweedie (`p=1.5`)** : 0이 자주 발생하고 양수 꼬리가 긴 매출량 분포에 적합. Poisson(p=1)과 Gamma(p=2) 사이의 혼합 분포

**하이퍼파라미터**
- `n_estimators=1200, learning_rate=0.03`
- `subsample=0.8, colsample_bytree=0.8` → 약한 정규화
- `random_state=42` → 재현성

In [ ]:
feature_cols = X_tr.columns.tolist()
models = {}
for h in range(1, HORIZON + 1):
    m = LGBMRegressor(
        objective='tweedie',
        tweedie_variance_power=1.5,
        n_estimators=1200,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        max_depth=-1,
        random_state=42,
    )
    tr_mask = (h_tr == h)
    va_mask = (h_va == h)
    m.fit(
        X_tr.loc[tr_mask, feature_cols], y_tr[tr_mask],
        sample_weight=w_tr[tr_mask],
        eval_set=[(X_va.loc[va_mask, feature_cols], y_va[va_mask])],
        eval_metric='rmse',
        callbacks=[],
    )
    models[h] = m
    print(f'  ✓ horizon={h} trained (n_train={tr_mask.sum():,})')

## 11. 검증용 SMAPE (참고)

윈도우 단위 점수라 대회 산식과 1:1로 일치하지 않지만, **모델 튜닝의 방향성**을 잡는 용도로 사용한다.

In [ ]:
def smape_equal_venue(y_true, y_pred, venues):
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, EPS, denom)
    sm = 2 * np.abs(y_true - y_pred) / denom
    df = pd.DataFrame({'v': venues, 'sm': sm})
    return df.groupby('v')['sm'].mean().mean()

## 12. 추론 — Direct 7-스텝 + 하이브리드 블렌딩

TEST 샘플의 28일을 그대로 입력으로 사용하고, 호라이즌 1~7을 **동시에(병렬로)** 예측한다.

최종 예측값은 세 가지 소스의 가중 평균:

$$ \hat{y}_h \;=\; W_{\text{MODEL}} \cdot y^{\text{LGBM}}_h \;+\; W_{\text{LAG7}} \cdot y^{\text{lag7}}_h \;+\; W_{\text{MEAN}} \cdot y^{\text{dow-mean}}_h $$

- `W_MODEL = 0.6` — LGBM 본체 예측
- `W_LAG7  = 0.3` — 단순 전주 동요일 값 (안정적 naive)
- `W_MEAN  = 0.1` — 28일 내 동요일 평균

마지막에 `max(ŷ, 1e-6)`으로 floor를 적용해 **SMAPE 분모 0** 을 회피한다.

In [ ]:
W_MODEL, W_LAG7, W_MEAN = 0.6, 0.3, 0.1

def predict_7days_direct(sample28_df: pd.DataFrame, menu_name: str):
    df = sample28_df.sort_values('영업일자').copy()
    df['매출수량'] = df['매출수량'].clip(lower=0)
    qty   = df['매출수량'].values.astype(float)
    dates = df['영업일자'].values
    assert len(qty) == LOOKBACK, f'expected {LOOKBACK} rows, got {len(qty)}'
    last_date = pd.to_datetime(df['영업일자'].max())
    base = window_features(qty)
    preds = []
    for h in range(1, HORIZON + 1):
        ctx  = horizon_context(dates, qty, last_date, h)
        X    = {**base, **ctx, 'menu_id': menu2id.get(menu_name, -1)}
        X_df = pd.DataFrame([X])[feature_cols]
        y_m  = float(models[h].predict(X_df)[0])
        y_n  = ctx['lag7_for_h']
        y_s  = ctx['same_dow_mean_h']
        y_hat = W_MODEL * y_m + W_LAG7 * y_n + W_MEAN * y_s
        y_hat = max(y_hat, EPS)
        preds.append(y_hat)
    return np.array(preds, dtype=float)

## 13. 제출 파일 생성

`test/` 폴더의 각 `TEST_XX.csv`에 대해 28일 시퀀스를 추출 → `predict_7days_direct`로 7일 예측 → `sample_submission.csv` 양식에 채워 저장한다.

**제출 파일명** : `submission_lgbm_{YYYYMMDD_HHMMSS}.csv` (실행 시각 타임스탬프 부여)

최고 성능을 달성한 버전: `submission_lgbm_20250821_221153.csv` (동일 폴더에 보관)

In [ ]:
sub = pd.read_csv(SAMPLE_SUB)
menu_cols = [c for c in sub.columns if c != '영업일자']

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.startswith('TEST_') and f.endswith('.csv')])
for tf in test_files:
    idx = int(tf.split('_')[1].split('.')[0])
    tdf = pd.read_csv(os.path.join(TEST_DIR, tf))
    tdf['영업일자'] = pd.to_datetime(tdf['영업일자'])
    tdf['매출수량'] = tdf['매출수량'].clip(lower=0)
    for menu in menu_cols:
        sdf = tdf.loc[tdf['영업장명_메뉴명'] == menu, ['영업일자', '영업장명_메뉴명', '매출수량']]
        if len(sdf) != LOOKBACK:
            continue
        preds7 = predict_7days_direct(sdf, menu)
        for h in range(1, HORIZON + 1):
            sub.loc[sub['영업일자'] == f'TEST_{idx:02d}+{h}일', menu] = preds7[h - 1]

for c in menu_cols:
    sub[c] = sub[c].clip(lower=0)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = os.path.join(OUTPUT_DIR, f'submission_lgbm_{timestamp}.csv')
sub.to_csv(out_path, index=False, encoding='utf-8')
print('[DONE] saved:', out_path)